# Apollo triage calibration
Source: `apollo-alerts-20261009.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('apollo-alerts-20261009.csv', encoding='utf-8-sig')
df['score'] = df['Triage Decision Score'].str.rstrip('%').astype(float) / 100
df['is_tp'] = df['SOAR Disposition'].str.startswith('True Positive')

# Labeled set: closed alerts with a real disposition
labeled = df[(df['SOAR Status'] == 'Closed') & ~df['SOAR Disposition'].isin(['Other', 'Not synced', 'No disposition yet'])].copy()
print(df.shape, labeled.shape)
df.head()

## Step 1 — What counts as benign
We only use the `labeled` alerts (closed, with a real disposition).
- **Benign** = anything that is *not* a True Positive (Benign Positive, False Positive, Undetermined)
- **TP** = True Positive — the alerts we must never auto-close

In [ ]:
labeled['is_benign'] = ~labeled['is_tp']
labeled['SOAR Disposition'].value_counts()

## Step 2 — Sweep the threshold
For each candidate threshold *t*: if we auto-close every alert with score below *t*, what do we get?
- `auto_closed` — how many alerts fall below *t*
- `pct_of_alerts` — share of all labeled alerts that would be auto-closed
- `tp_missed` — True Positives that would have been auto-closed (mistakes)
- `tp_rate` — `tp_missed / auto_closed`; this is what must stay ≤ 0.5%
- `bp_rate` — share of the auto-closed alerts that are specifically Benign Positive

In [ ]:
thresholds = np.round(np.arange(0.01, 0.51, 0.01), 2)  # 1% to 50%
rows = []
for t in thresholds:
    below = labeled[labeled['score'] < t]
    n = len(below)
    tp = below['is_tp'].sum()
    bp = below['SOAR Disposition'].str.startswith('Benign Positive').sum()
    rows.append({'threshold': t, 'auto_closed': n, 'pct_of_alerts': n / len(labeled),
                 'tp_missed': tp, 'tp_rate': tp / n, 'bp_rate': bp / n})
sweep = pd.DataFrame(rows)
sweep.style.format({'threshold': '{:.0%}', 'pct_of_alerts': '{:.1%}', 'tp_rate': '{:.2%}', 'bp_rate': '{:.1%}'})

## Step 3 — Highest threshold that stays within 0.5% TP
Pick the largest threshold where `tp_rate` ≤ 0.5%.

In [ ]:
TOLERANCE = 0.005
ok = sweep[sweep['tp_rate'] <= TOLERANCE]
best = ok.iloc[-1]
print(f"Threshold: score < {best['threshold']:.0%}")
print(f"Auto-closed: {int(best['auto_closed'])} alerts ({best['pct_of_alerts']:.1%} of labeled)")
print(f"TPs missed: {int(best['tp_missed'])} ({best['tp_rate']:.2%})")
print(f"Benign Positive share: {best['bp_rate']:.1%}")

## Step 4 — Picture of the sweep

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sweep['threshold'] * 100, sweep['tp_rate'] * 100, marker='.', label='TP rate among auto-closed')
ax.axhline(TOLERANCE * 100, color='red', linestyle='--', label='0.5% tolerance')
ax.axvline(best['threshold'] * 100, color='gray', linestyle=':', label=f"chosen threshold ({best['threshold']:.0%})")
ax.set_xlabel('Threshold (score %)')
ax.set_ylabel('TP rate (%)')
ax.legend()
plt.show()

## Step 5 — AI agent queue: Benign Positive rate by score
Only alerts routed to the **AI agent**. Target is strictly **Benign Positive**.
- `band_*` — alerts whose score falls *in* that 1%-wide band (e.g. 3%–4%)
- `cum_*` — all AI agent alerts with score *below* the band's upper edge

In [ ]:
ai = labeled[labeled['Triage Queue'] == 'AI agent'].copy()
ai['is_bp'] = ai['SOAR Disposition'].str.startswith('Benign Positive')

edges = np.round(np.arange(0, 0.51, 0.01), 2)
rows = []
for lo, hi in zip(edges[:-1], edges[1:]):
    band = ai[(ai['score'] >= lo) & (ai['score'] < hi)]
    cum = ai[ai['score'] < hi]
    rows.append({'score_band': f'{lo:.0%}-{hi:.0%}',
                 'band_alerts': len(band), 'band_bp_rate': band['is_bp'].mean(),
                 'cum_alerts': len(cum), 'cum_bp_rate': cum['is_bp'].mean()})
ai_bp = pd.DataFrame(rows)
print(f"AI agent labeled alerts: {len(ai)}, Benign Positive: {ai['is_bp'].sum()} ({ai['is_bp'].mean():.1%})")
ai_bp.style.format({'band_bp_rate': '{:.1%}', 'cum_bp_rate': '{:.1%}'})

## Step 6 — Visualize: AI agent queue, Benign Positive by score
- **Top:** Benign Positive rate in each 1% score band (bars) and cumulatively below each score (line)
- **Bottom:** how many AI agent alerts fall in each band — so you can see how much volume each rate is based on

In [ ]:
BLUE, ORANGE, INK, MUTED, GRID = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e', '#e6e5e0'

x = np.arange(len(ai_bp))  # band index: 0 = 0-1%, 1 = 1-2%, ...
fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                                  gridspec_kw={'height_ratios': [3, 1.2]})

top.bar(x, ai_bp['band_bp_rate'] * 100, width=0.8, color=BLUE, label='Benign Positive rate in band')
top.plot(x, ai_bp['cum_bp_rate'] * 100, color=ORANGE, linewidth=2, marker='o', markersize=4,
         label='Benign Positive rate below score (cumulative)')
top.set_ylim(0, 100)
top.set_ylabel('Benign Positive (%)', color=MUTED)
top.set_title('AI agent queue — Benign Positive rate by triage score', loc='left', color=INK, pad=28)
top.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)

bottom.bar(x, ai_bp['band_alerts'], width=0.8, color=MUTED)
bottom.set_ylabel('Alerts in band', color=MUTED)
bottom.set_xlabel('Triage Decision Score (%)', color=MUTED)
bottom.set_xticks(x[::5], [f'{i}%' for i in x[::5]])

for ax in (top, bottom):
    ax.grid(axis='y', color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color(GRID)
    ax.tick_params(colors=MUTED)

plt.tight_layout()
plt.show()

## Step 7 — Recommendation: auto-close AI agent alerts with score < 10%
Step 3 picked 12% using the **observed** TP rate. But the observed rate is from one sample — the true rate could be higher.
Here we also compute a **95% worst case** (upper confidence bound) for the TP rate: we are 95% confident the real TP rate is no higher than this.

Rule: pick a threshold where even the **worst case** stays ≤ 0.5%.

In [ ]:
from scipy.stats import beta

def worst_case_tp_rate(tp, n, confidence=0.95):
    # Clopper-Pearson upper bound on the TP rate
    return beta.ppf(confidence, tp + 1, n - tp)

rows = []
for t in [0.10, 0.11, 0.12]:
    below = labeled[labeled['score'] < t]
    n, tp = len(below), below['is_tp'].sum()
    rows.append({'threshold': t, 'auto_closed': n, 'tp_missed': tp, 'tp_rate': tp / n,
                 'worst_case_tp_rate': worst_case_tp_rate(tp, n),
                 'bp_rate': below['SOAR Disposition'].str.startswith('Benign Positive').mean()})
compare = pd.DataFrame(rows)
compare['within_tolerance'] = compare['worst_case_tp_rate'] <= TOLERANCE
compare.style.format({'threshold': '{:.0%}', 'tp_rate': '{:.2%}', 'worst_case_tp_rate': '{:.2%}', 'bp_rate': '{:.1%}'})

### What gets auto-closed at < 10%

In [ ]:
RECOMMENDED = 0.10
auto = labeled[labeled['score'] < RECOMMENDED]
print(f"Auto-closed: {len(auto)} alerts ({len(auto) / len(labeled):.1%} of labeled, "
      f"{len(auto) / len(ai):.1%} of AI agent queue)")
print(f"TPs missed: {auto['is_tp'].sum()} ({auto['is_tp'].mean():.2%}), "
      f"worst case {worst_case_tp_rate(auto['is_tp'].sum(), len(auto)):.2%}")
auto['SOAR Disposition'].value_counts()

In [ ]:
# The True Positives that would be auto-closed
auto[auto['is_tp']][['Severity', 'Customer', 'Rule Name', 'Triage Decision Score', 'Container ID', 'Created']]

### Open items before relying on this
1. ~3.3% of auto-closed alerts are **Undetermined or False Positive**, not Benign Positive — decide how auto-close should label them.
2. The threshold was chosen on all data (Aug–Oct). Not yet checked on **newer data** (e.g. pick on Aug–Sep, test on Oct).

## Step 8 — Test the 10% threshold on October
The 10% threshold was chosen using all data. Here we check whether it still holds when we look at **October on its own**,
and compare it with the earlier months (Aug–Sep).

The month comes from the first word of `Created` (e.g. `Aug 3, 15:50 EDT` → `Aug`).

In [ ]:
labeled['month'] = labeled['Created'].str.split().str[0]

periods = {
    'Aug': labeled[labeled['month'] == 'Aug'],
    'Sep': labeled[labeled['month'] == 'Sep'],
    'Aug–Sep': labeled[labeled['month'].isin(['Aug', 'Sep'])],
    'Oct': labeled[labeled['month'] == 'Oct'],
}
rows = []
for name, period in periods.items():
    below = period[period['score'] < RECOMMENDED]
    n, tp = len(below), below['is_tp'].sum()
    rows.append({'period': name, 'labeled_alerts': len(period), 'auto_closed': n,
                 'pct_auto_closed': n / len(period), 'tp_missed': tp, 'tp_rate': tp / n,
                 'worst_case_tp_rate': worst_case_tp_rate(tp, n),
                 'bp_rate': below['SOAR Disposition'].str.startswith('Benign Positive').mean()})
by_period = pd.DataFrame(rows)
by_period['within_tolerance'] = by_period['worst_case_tp_rate'] <= TOLERANCE
by_period.style.format({'pct_auto_closed': '{:.1%}', 'tp_rate': '{:.2%}',
                        'worst_case_tp_rate': '{:.2%}', 'bp_rate': '{:.1%}'})

### October alerts not yet labeled
October is still in progress. Alerts that are still Open / Escalated / Preparing have no final disposition yet,
so they are **not** in the test above.

In [ ]:
pd.crosstab(df['Created'].str.split().str[0], df['SOAR Status'])

## Step 9 — Visualize: the 10% threshold inside the AI agent queue
Same chart as Step 6 (AI agent queue only, scores 0–50%), with the threshold added:
- **Shaded area (< 10%)** — alerts that would be **auto-closed**
- **Unshaded (10%–50%)** — alerts that stay with the **AI agent** for review
- Labels show how many alerts fall on each side and their Benign Positive rate

In [ ]:
SHADE = '#f0efe9'
t_idx = int(round(RECOMMENDED * 100))  # band index where the threshold sits

auto_side = ai[ai['score'] < RECOMMENDED]
review_side = ai[ai['score'] >= RECOMMENDED]

fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 7.5), sharex=True,
                                  gridspec_kw={'height_ratios': [3, 1.2]})

for ax in (top, bottom):
    ax.axvspan(-0.5, t_idx - 0.5, color=SHADE, zorder=0)
    ax.axvline(t_idx - 0.5, color=INK, linestyle='--', linewidth=1.5)

top.bar(x, ai_bp['band_bp_rate'] * 100, width=0.8, color=BLUE, label='Benign Positive rate in band')
top.plot(x, ai_bp['cum_bp_rate'] * 100, color=ORANGE, linewidth=2, marker='o', markersize=4,
         label='Benign Positive rate below score (cumulative)')
top.set_ylim(0, 115)
top.set_yticks(range(0, 101, 20))
top.set_ylabel('Benign Positive (%)', color=MUTED)
top.set_title(f'AI agent queue — {RECOMMENDED:.0%} auto-close threshold', loc='left', color=INK, pad=28)
top.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)

top.text((t_idx - 1) / 2, 107,
         f"AUTO-CLOSE  (< {RECOMMENDED:.0%})\n{len(auto_side):,} alerts · {auto_side['is_bp'].mean():.1%} BP",
         ha='center', va='center', color=INK, fontsize=10)
top.text((t_idx + len(x) - 1) / 2, 107,
         f"AI AGENT REVIEW  ({RECOMMENDED:.0%}–50%)\n{len(review_side):,} alerts · {review_side['is_bp'].mean():.1%} BP",
         ha='center', va='center', color=INK, fontsize=10)

bottom.bar(x, ai_bp['band_alerts'], width=0.8, color=MUTED)
bottom.set_ylabel('Alerts in band', color=MUTED)
bottom.set_xlabel('Triage Decision Score (%)', color=MUTED)
bottom.set_xticks(x[::5], [f'{i}%' for i in x[::5]])

for ax in (top, bottom):
    ax.grid(axis='y', color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color(GRID)
    ax.tick_params(colors=MUTED)

plt.tight_layout()
plt.show()

## Step 10 — True Positives: is there a score above which ~95%+ are TP?
Mirror of Step 2, from the top down. For each threshold *t*: take every labeled alert with score **≥ t**.
- `alerts_above` — how many alerts score at or above *t*
- `tp_rate` — share of them that are True Positive (goal: ≥ 95%)
- `worst_case_tp_rate` — 95% lower bound: we are 95% confident the real TP rate is at least this
- `tp_below` — True Positives that score *below* *t* (not captured by this threshold)

In [ ]:
TP_TARGET = 0.95

def worst_case_low(tp, n, confidence=0.95):
    # Clopper-Pearson lower bound on the TP rate
    return beta.ppf(1 - confidence, tp, n - tp + 1) if tp else 0.0

rows = []
for t in np.round(np.arange(0.50, 0.98, 0.02), 2):
    above = labeled[labeled['score'] >= t]
    n, tp = len(above), above['is_tp'].sum()
    rows.append({'threshold': t, 'alerts_above': n, 'tp': tp, 'tp_rate': tp / n,
                 'worst_case_tp_rate': worst_case_low(tp, n),
                 'tp_below': labeled['is_tp'].sum() - tp})
tp_sweep = pd.DataFrame(rows)
print(f"Highest TP rate reached: {tp_sweep['tp_rate'].max():.1%} "
      f"(at score >= {tp_sweep.loc[tp_sweep['tp_rate'].idxmax(), 'threshold']:.0%})")
print(f"Thresholds reaching {TP_TARGET:.0%}: {(tp_sweep['tp_rate'] >= TP_TARGET).sum()}")
tp_sweep.style.format({'threshold': '{:.0%}', 'tp_rate': '{:.1%}', 'worst_case_tp_rate': '{:.1%}'})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tp_sweep['threshold'] * 100, tp_sweep['tp_rate'] * 100, color=BLUE, linewidth=2, marker='o',
        markersize=4, label='TP rate at or above score')
ax.axhline(TP_TARGET * 100, color=INK, linestyle='--', linewidth=1.5, label=f'{TP_TARGET:.0%} target')
ax.set_ylim(0, 100)
ax.set_xlabel('Threshold (score %)', color=MUTED)
ax.set_ylabel('True Positive (%)', color=MUTED)
ax.set_title('Share of True Positives among alerts at or above each score', loc='left', color=INK, pad=28)
ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
ax.grid(axis='y', color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color(GRID)
ax.tick_params(colors=MUTED)
plt.show()

In [ ]:
# What the top-scoring alerts (>= 90%) actually are
labeled[labeled['score'] >= 0.90]['SOAR Disposition'].value_counts()

### Step 10 findings — no score reaches 95% True Positive

| Score ≥ | Alerts | TPs | TP rate | TPs scoring below |
|---|---|---|---|---|
| 50% (analyst queue) | 1,432 | 1,078 | 75.3% | 421 |
| 70% | 824 | 667 | 80.9% | 832 |
| **80% (peak)** | **404** | **340** | **84.2%** | 1,159 |
| 90% | 69 | 57 | 82.6% | 1,442 |
| 96% | 14 | 10 | 71.4% | 1,489 |

- **Tops out at ~84%** around score 80%, then dips; above ~92% it falls to ~72% (few alerts there, so noisy).
- **Highest-scoring alerts aren't pure TPs:** of the 69 alerts scoring ≥ 90%, 57 TP, 7 Undetermined, 5 Benign Positive.
- **Raising the threshold costs a lot:** 50% → 80% only lifts TP rate 75% → 84%, but TPs left below go from 421 to 1,159.

**Takeaway:** the score is reliable at the **bottom** (supports auto-closing benign alerts, Steps 7–8) but not at the **top** —
no single threshold can auto-confirm True Positives at 95%+.

**Possible next steps**
1. Check high scores **per rule** — some rules may hit 95%+ even if the overall rate doesn't.
2. Look at the **12 non-TP alerts scoring ≥ 90%** to see why they scored high.

## Step 11 — Reliability diagrams: True Positive and Benign Positive
A reliability diagram checks whether the score means what it says.
- Alerts are grouped into **10% score bands** (0–10%, 10–20%, …).
- Each dot sits at the band's **average score** (x) and the share of alerts in that band that **actually** were TP or BP (y).
- The bars underneath show how many alerts are in each band — dots with few alerts behind them are less reliable.

**Reference lines (dashed):** we treat the score as the chance an alert is a True Positive.
- **TP diagram:** perfect score → dots on the diagonal (score 30% → 30% are TP). Dot **below** = score too high; **above** = score too low.
- **BP diagram:** the line is `1 − score`, i.e. the BP rate if *every* non-TP alert were Benign Positive. Dots sit a little
  below it because some non-TP alerts are False Positive or Undetermined instead.

In [ ]:
bins = np.round(np.arange(0, 1.01, 0.10), 2)
labeled['band'] = pd.cut(labeled['score'], bins=bins, right=False, include_lowest=True)
labeled['is_bp'] = labeled['SOAR Disposition'].str.startswith('Benign Positive')

reliability = labeled.groupby('band', observed=False).agg(
    alerts=('score', 'size'),
    avg_score=('score', 'mean'),
    tp_rate=('is_tp', 'mean'),
    bp_rate=('is_bp', 'mean'),
).reset_index()
reliability['band'] = [f'{b.left:.0%}–{b.right:.0%}' for b in reliability['band']]
reliability.style.format({'avg_score': '{:.1%}', 'tp_rate': '{:.1%}', 'bp_rate': '{:.1%}'})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
panels = [
    (0, 'tp_rate', ORANGE, 'True Positive', lambda s: s, 'Perfect calibration (rate = score)'),
    (1, 'bp_rate', BLUE, 'Benign Positive', lambda s: 1 - s, 'If every non-TP were BP (rate = 1 − score)'),
]
line_x = np.linspace(0, 1, 101)

for col, rate_col, color, name, ref, ref_label in panels:
    top, bottom = axes[0, col], axes[1, col]
    top.plot(line_x * 100, ref(line_x) * 100, color=MUTED, linestyle='--', linewidth=1.2, label=ref_label)
    top.plot(reliability['avg_score'] * 100, reliability[rate_col] * 100, color=color, linewidth=2,
             marker='o', markersize=8, label=f'Actual {name} rate')
    top.set_ylim(0, 100)
    top.set_ylabel(f'Actual {name} (%)', color=MUTED)
    top.set_title(f'Reliability — {name}', loc='left', color=INK, pad=28)
    top.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, fontsize=9)

    bottom.bar(reliability.index * 10 + 5, reliability['alerts'], width=8, color=MUTED)
    bottom.set_ylabel('Alerts in band', color=MUTED)
    bottom.set_xlabel('Triage Decision Score (%)', color=MUTED)
    bottom.set_xticks(range(0, 101, 10))

    for ax in (top, bottom):
        ax.set_xlim(0, 100)
        ax.grid(axis='y', color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['left', 'bottom']].set_color(GRID)
        ax.tick_params(colors=MUTED)

plt.tight_layout()
plt.show()

### Step 11 findings

| Score band | Alerts | Avg score | Actual TP | Actual BP |
|---|---|---|---|---|
| 0–10% | 4,227 | 4.1% | 0.2% | 96.5% |
| 10–20% | 1,198 | 14.1% | 4.7% | 85.6% |
| 20–30% | 845 | 24.4% | 10.9% | 76.1% |
| 30–40% | 421 | 34.9% | 28.5% | 56.3% |
| 40–50% | 316 | 44.8% | 45.3% | 42.1% |
| 50–60% | 261 | 55.0% | 63.6% | 24.1% |
| 60–70% | 347 | 65.2% | 70.6% | 12.4% |
| 70–80% | 420 | 75.2% | 77.9% | 11.7% |
| 80–90% | 335 | 84.1% | 84.5% | 9.6% |
| 90–100% | 69 | 93.9% | 82.6% | 7.2% |

**True Positive**
- **Below 30%:** actual TP rate is *lower* than the score (e.g. 4.1% score → 0.2% TP). The score is cautious here — good for auto-closing.
- **40–90%:** dots sit close to or slightly above the diagonal — the score is roughly honest, if anything a bit low.
- **90–100%:** dot drops *below* the diagonal (93.9% score → 82.6% TP) — the score is overconfident at the top. Only 69 alerts, so less certain.

**Benign Positive**
- **Below 30%:** dots sit almost exactly on the `1 − score` line — nearly every non-TP alert here is Benign Positive.
- **30–70%:** dots fall clearly below the line. Two reasons: more alerts are Undetermined / False Positive here,
  and from 50% up the actual TP rate is *higher* than the score (see TP diagram), leaving less room for BP.
- **Above 60%:** BP levels off around 7–12%.

*All data (Aug–Oct) combined. Step 8 showed October behaves differently — not reflected here.*